سلول ۱ — import

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold

from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    brier_score_loss,
)

from xgboost import XGBClassifier

from tqdm.auto import tqdm

سلول ۲ — مسیرها

In [ ]:
PROJECT_ROOT = Path(".")

DATA_ML_DIR = PROJECT_ROOT / "Data_ml"
QC_DIR = PROJECT_ROOT / "Data_proc" / "qc_reports"

PAIR_FEATURE_DIR = DATA_ML_DIR / "pair_features"

RESULT_DIR = DATA_ML_DIR / "baseline_results"

RESULT_DIR.mkdir(parents=True, exist_ok=True)

سلول ۳ — مدل‌ها

In [ ]:
MODELS = [
    "esm2_t6_8M_UR50D",
    "esm2_t12_35M_UR50D",
    "esm2_t30_150M_UR50D",
    "esm2_t33_650M_UR50D",
    "esm2_t36_3B_UR50D",
    "prot_bert",
    "prot_bert_bfd",
]

FEATURE_TYPES = [
    "per_sequence",
    "per_residue",
    "fusion",
]

سلول ۴ — loader

In [ ]:
def load_feature_set(model_name, feature_type):

    model_dir = PAIR_FEATURE_DIR / model_name

    meta = pd.read_csv(
        model_dir / "meta.csv",
        dtype=str,
        low_memory=False
    )

    X = pd.read_parquet(
        model_dir / f"{feature_type}.features.parquet"
    )

    y = meta["label"].astype(int).values

    groups = meta["group_id"].values

    return X, y, groups

سلول ۵ — evaluator

In [ ]:
def evaluate_model(
    estimator,
    X,
    y,
    groups,
    n_splits=5
):

    gkf = GroupKFold(
        n_splits=n_splits
    )

    fold_rows = []

    for fold_idx, (tr, te) in enumerate(
        gkf.split(X, y, groups)
    ):

        Xtr = X.iloc[tr]
        Xte = X.iloc[te]

        ytr = y[tr]
        yte = y[te]

        estimator.fit(
            Xtr,
            ytr
        )

        prob = estimator.predict_proba(Xte)[:,1]

        pred = (prob >= 0.5).astype(int)

        fold_rows.append({

            "fold": fold_idx,

            "roc_auc":
                roc_auc_score(yte, prob),

            "pr_auc":
                average_precision_score(yte, prob),

            "f1":
                f1_score(yte, pred),

            "accuracy":
                accuracy_score(yte, pred),

            "precision":
                precision_score(yte, pred),

            "recall":
                recall_score(yte, pred),

            "brier":
                brier_score_loss(yte, prob),
        })

    return pd.DataFrame(
        fold_rows
    )

سلول ۶ — Logistic

In [ ]:
def build_logistic():

    return LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        solver="liblinear",
        random_state=42,
    )

سلول ۷ — Random Forest

In [ ]:
def build_rf():

    return RandomForestClassifier(

        n_estimators=500,

        max_depth=None,

        min_samples_leaf=1,

        class_weight="balanced",

        n_jobs=-1,

        random_state=42,
    )

سلول ۸ — XGBoost

In [ ]:
def build_xgb():

    return XGBClassifier(

        n_estimators=1000,

        max_depth=6,

        learning_rate=0.05,

        subsample=0.8,

        colsample_bytree=0.8,

        eval_metric="logloss",

        tree_method="hist",

        random_state=42,
    )

سلول ۹ — اجرای کامل ۶۳ benchmark

In [ ]:
all_results = []

for model_name in MODELS:

    for feature_type in FEATURE_TYPES:

        print()
        print("="*100)
        print(model_name)
        print(feature_type)

        X,y,groups = load_feature_set(
            model_name,
            feature_type,
        )

        estimators = {

            "logistic":
                build_logistic(),

            "random_forest":
                build_rf(),

            "xgboost":
                build_xgb(),
        }

        for algo_name, est in estimators.items():

            print("running:", algo_name)

            fold_df = evaluate_model(
                est,
                X,
                y,
                groups,
            )

            result_row = {

                "embedding_model":
                    model_name,

                "feature_type":
                    feature_type,

                "algorithm":
                    algo_name,
            }

            for metric in [

                "roc_auc",
                "pr_auc",
                "f1",
                "accuracy",
                "precision",
                "recall",
                "brier",
            ]:

                result_row[f"{metric}_mean"] = \
                    fold_df[metric].mean()

                result_row[f"{metric}_std"] = \
                    fold_df[metric].std()

            all_results.append(
                result_row
            )

results_df = pd.DataFrame(
    all_results
)

display(
    results_df
)

results_df.to_csv(
    RESULT_DIR / "baseline_results_all.csv",
    index=False
)

سلول ۱۰ — رتبه‌بندی نهایی

In [ ]:
leaderboard = (
    results_df
    .sort_values(
        "pr_auc_mean",
        ascending=False
    )
)

display(
    leaderboard.head(20)
)

leaderboard.to_csv(
    RESULT_DIR / "leaderboard_pr_auc.csv",
    index=False
)